In [22]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

pd.set_option('display.max_columns', None)

In [23]:
# Load dataset
data_path = "data/cleaned_sales_data.csv"
if not os.path.exists(data_path) and os.path.exists("../data/cleaned_sales_data.csv"):
    data_path = "../data/cleaned_sales_data.csv"

df = pd.read_csv(data_path, low_memory=False)

# Remove outliers
df = df[(df["ClosePrice"] >= 100000) & (df["ClosePrice"] <= 5000000)].copy()
df["CloseDate"] = pd.to_datetime(df["CloseDate"])

In [24]:
# 1. Bed / Bath ratio
df["bed_bath_ratio"] = df["BedroomsTotal"] / (df["BathroomsTotalInteger"].fillna(1) + 1)

# 2. Property age in years
current_year = 2026
df["property_age"] = current_year - df["YearBuilt"]
# Fill missing age with median
df["property_age"] = df["property_age"].apply(lambda x: x if 0 <= x <= 150 else np.nan)
df["property_age"] = df["property_age"].fillna(df["property_age"].median())

print("Basic Engineered Features Created successfully!")

Basic Engineered Features Created successfully!


In [25]:
import glob

shapefile_matches = glob.glob("data/school_districts/*.shp") + glob.glob("../data/school_districts/*.shp")

if not shapefile_matches:
    raise FileNotFoundError("Could not find the .shp file in data/school_districts/. Please check folder placement!")

shapefile_path = shapefile_matches[0]
print(f"Loading shapefile from: {shapefile_path}")

school_gdf = gpd.read_file(shapefile_path)

df_geo = df.dropna(subset=["Latitude", "Longitude"]).copy()

geometry = [Point(xy) for xy in zip(df_geo["Longitude"], df_geo["Latitude"])]
properties_gdf = gpd.GeoDataFrame(df_geo, geometry=geometry, crs="EPSG:4326")

school_gdf = school_gdf.to_crs(properties_gdf.crs)

district_col = [col for col in school_gdf.columns if "NAME" in col.upper() or "DISTRICT" in col.upper()][0]
joined_gdf = gpd.sjoin(properties_gdf, school_gdf[[district_col, "geometry"]], how="left", predicate="within")

joined_gdf = joined_gdf[~joined_gdf.index.duplicated(keep="first")]

df["school_district"] = "Unknown"
df.loc[joined_gdf.index, "school_district"] = joined_gdf[district_col].fillna("Unknown")

district_means = df.groupby("school_district")["ClosePrice"].transform("mean")
df["school_district_avg_price"] = district_means

print("Spatial join complete! School district price layer added successfully.")

Loading shapefile from: ../data/school_districts/DistrictAreas2425.shp
Spatial join complete! School district price layer added successfully.


In [26]:
new_features = ["bed_bath_ratio", "property_age", "school_district_avg_price"]

scaler = StandardScaler()
df[[f"{col}_scaled" for col in new_features]] = scaler.fit_transform(df[new_features])

feature_cols = [
    "LivingArea_scaled", "BedroomsTotal_scaled", 
    "BathroomsTotalInteger_scaled", "LotSizeSquareFeet_scaled",
    "bed_bath_ratio_scaled", "property_age_scaled", "school_district_avg_price_scaled"
]

# Time-based Train/Test Split
test_mask = (df["CloseDate"].dt.year == 2026) & (df["CloseDate"].dt.month == 5)
test_df = df[test_mask].copy()

test_start = pd.Timestamp("2026-05-01")
train_start = test_start - pd.DateOffset(months=6)
train_df = df[(df["CloseDate"] >= train_start) & (df["CloseDate"] < test_start)].copy()

X_train, y_train = train_df[feature_cols], train_df["ClosePrice"]
X_test, y_test = test_df[feature_cols], test_df["ClosePrice"]

# Train Models
lr = LinearRegression().fit(X_train, y_train)
rf = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1).fit(X_train, y_train)

lr_r2 = r2_score(y_test, lr.predict(X_test))
rf_r2 = r2_score(y_test, rf.predict(X_test))

# Comparison
summary_df = pd.DataFrame({
    "Model": ["Linear Regression (Baseline + New Features)", "Random Forest (With New Features)"],
    "Test R2 Score": [lr_r2, rf_r2]
})

print("--- Week 6 Feature Engineering Performance Summary ---")
print(summary_df.to_string(index=False))

--- Week 6 Feature Engineering Performance Summary ---
                                      Model  Test R2 Score
Linear Regression (Baseline + New Features)       0.601537
          Random Forest (With New Features)       0.675821
